# Notebook 4: DistilBERT fine-tuning

**Research question:** Can a spam classifier trained on SMS messages generalize to emails?

This notebook fine-tunes [DistilBERT](https://arxiv.org/abs/1910.01108) separately on SMS and Enron, then compares it with TF-IDF + Logistic Regression and TextCNN under the same transfer protocol.

## Experiment design

- Fine-tune `distilbert-base-uncased` separately on SMS and Enron.
- Reuse the existing train, validation, and test splits.
- Truncate inputs to 256 tokens and pad dynamically within each batch.
- Derive balanced class weights from each training split.
- Keep the initial benchmark configuration fixed between domains and select checkpoints by validation F1.
- Evaluate both fitted models on both test domains; test labels are not used for fitting, checkpoint selection, or threshold tuning.

The outputs currently shown are the initial benchmark with training seed `42`. The robustness extension keeps split seed `42` fixed, selects hyperparameters by source-validation macro-F1, and evaluates the locked configuration with training seeds `13`, `42`, `73`, `101`, and `137`. SMS -> Enron is the primary transfer direction; Enron -> SMS is a secondary reverse-direction check. The already inspected test sets are fixed confirmation benchmarks and remain excluded from all selection decisions.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/kbozukov-coke/cross-domain-spam-detection.git'
PROJECT_NAME = 'cross-domain-spam-detection'
IS_KAGGLE = Path('/kaggle/working').exists()

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working') / PROJECT_NAME
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'],
            check=True,
        )
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install', '-q',
            'transformers>=4.57,<5', 'accelerate>=1.10,<2',
        ],
        check=True,
    )
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import transformers
from sklearn.metrics import ConfusionMatrixDisplay

from src.data import load_prepared_splits, summarize_splits
from src.distilbert import run_distilbert_experiments
from src.modeling import run_transfer_experiments
from src.protocol import DATA_SPLIT_SEED, REFERENCE_TRAINING_SEED

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)

print(f'Running in: {"Kaggle" if IS_KAGGLE else "local environment"}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(f'GPU devices: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}')

if IS_KAGGLE and not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator before running this notebook.')

## Load the prepared data

The preprocessing and splits are identical to the previous notebooks.

In [ ]:
splits, _ = load_prepared_splits(random_state=DATA_SPLIT_SEED)
split_summary = summarize_splits(splits)
split_summary.loc[:, ['dataset', 'split', 'rows', 'ham', 'spam', 'spam_rate']]

## Initial benchmark configuration

This seed-42, two-epoch configuration is retained as the initial reference and is used for both domains. Batch sizes are per device; the effective training batch size is recorded with the results.

In [ ]:
DISTILBERT_CONFIG = {
    'model_name': 'distilbert-base-uncased',
    'random_state': REFERENCE_TRAINING_SEED,
    'max_length': 256,
    'learning_rate': 2e-5,
    'epochs': 2,
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 16,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'require_gpu': True,
}

pd.Series(DISTILBERT_CONFIG, name='value').to_frame()

## Fine-tune on each domain

This is the only long-running cell. Models are trained and evaluated sequentially so the SMS model is released before the Enron model is created. The classification head is task-specific and is therefore initialized from scratch.

In [ ]:
training_histories, distilbert_results, distilbert_details = (
    run_distilbert_experiments(splits, **DISTILBERT_CONFIG)
)
print('Finished fine-tuning the SMS and Enron DistilBERT models.')

## Learning curves

Two epochs provide only an early diagnostic, not evidence of full convergence. SMS validation loss remains nearly flat while training loss falls. For Enron, validation loss rises from 0.030 to 0.039 in epoch 2 while training loss continues to fall, indicating the beginning of overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for axis, domain in zip(axes, ['sms', 'enron']):
    history = training_histories[domain]
    train_rows = history.dropna(subset=['loss'])
    validation_rows = history.dropna(subset=['eval_loss'])
    axis.plot(train_rows['epoch'], train_rows['loss'], marker='o', label='train')
    axis.plot(
        validation_rows['epoch'],
        validation_rows['eval_loss'],
        marker='o',
        label='validation',
    )
    axis.set_title(f'{domain.upper()} fine-tuning loss')
    axis.set_xlabel('Epoch')
    axis.set_ylabel('Cross-entropy')
    axis.legend()

plt.tight_layout()
plt.show()

## DistilBERT results

In-domain F1 is high for both SMS (0.974) and Enron (0.991), but cross-domain F1 falls to 0.557 for SMS -> Enron and 0.268 for Enron -> SMS. Precision, recall, and ROC-AUC show how the error profile changes across domains.

In [ ]:
metric_columns = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
display_results = distilbert_results.copy()
display_results[metric_columns] = display_results[metric_columns].round(3)
display_results['training_seconds'] = display_results['training_seconds'].round(1)
display_results.loc[:, [
    'train_domain', 'test_domain', 'setting', 'test_rows',
    *metric_columns, 'epochs_trained', 'training_seconds', 'parameters',
    'gpu_count', 'effective_train_batch_size',
]]

## Compare all three approaches

TF-IDF + Logistic Regression is refitted on the same splits. TextCNN metrics are loaded from the versioned artifact produced by [Notebook 3](https://www.kaggle.com/code/kaloyanbozukov/notebook-3-textcnn-cross-domain-experiment), avoiding another training run. All three approaches use the same test rows and metrics; small score differences are descriptive because only one run is available.

In [ ]:
_, baseline_results = run_transfer_experiments(
    splits, random_state=REFERENCE_TRAINING_SEED
)
baseline_results = baseline_results.assign(model='TF-IDF + Logistic Regression')

textcnn_results_path = PROJECT_ROOT / 'results' / 'textcnn_results.csv'
textcnn_reference = pd.read_csv(textcnn_results_path)
required_reference_columns = {
    'model', 'train_domain', 'test_domain', *metric_columns,
}
missing_reference_columns = required_reference_columns - set(textcnn_reference.columns)
if missing_reference_columns:
    raise ValueError(
        f'Missing TextCNN result columns: {sorted(missing_reference_columns)}'
    )

comparison = pd.concat(
    [
        baseline_results.loc[:, ['model', 'train_domain', 'test_domain', *metric_columns]],
        textcnn_reference.loc[:, ['model', 'train_domain', 'test_domain', *metric_columns]],
        distilbert_results.loc[:, ['model', 'train_domain', 'test_domain', *metric_columns]],
    ],
    ignore_index=True,
)
comparison['experiment'] = (
    comparison['train_domain'].str.upper()
    + ' -> '
    + comparison['test_domain'].str.upper()
)
experiment_order = [
    'SMS -> SMS', 'SMS -> ENRON', 'ENRON -> SMS', 'ENRON -> ENRON'
]
comparison.pivot(index='experiment', columns='model', values='f1').reindex(
    experiment_order
).round(3)

In [ ]:
plt.figure(figsize=(11, 5))
axis = sns.barplot(
    data=comparison,
    x='experiment',
    y='f1',
    hue='model',
    order=experiment_order,
)
axis.set_title('F1 comparison across models and domains')
axis.set_xlabel('Train -> test domain')
axis.set_ylabel('F1 score')
axis.set_ylim(0, 1)
axis.legend(title='Model', loc='lower right')
for container in axis.containers:
    axis.bar_label(container, fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()

## DistilBERT confusion matrices

Domain transfer shifts both models toward predicting spam. SMS -> Enron produces 825 false positives and 291 false negatives; Enron -> SMS produces 478 false positives and only 7 false negatives. The latter therefore has high recall (0.927) but very low precision (0.157).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for axis, row in zip(axes.ravel(), distilbert_results.itertuples(index=False)):
    details = distilbert_details[(row.train_domain, row.test_domain)]
    ConfusionMatrixDisplay.from_predictions(
        details['label'],
        details['prediction'],
        display_labels=['ham', 'spam'],
        colorbar=False,
        ax=axis,
    )
    axis.set_title(
        f'Train {row.train_domain.upper()} -> Test {row.test_domain.upper()}'
    )

plt.tight_layout()
plt.show()

## SMS -> Enron error analysis

The five most confident errors illustrate failure cases of the SMS-trained model on email. They are a selected qualitative sample; the confusion matrix and aggregate metrics provide the quantitative evidence.

In [ ]:
sms_to_enron = distilbert_details[('sms', 'enron')].copy()
cross_domain_errors = sms_to_enron.loc[~sms_to_enron['correct']].copy()
cross_domain_errors['error_type'] = cross_domain_errors['label'].map(
    {0: 'false positive', 1: 'false negative'}
)
cross_domain_errors['confidence'] = (
    cross_domain_errors['spam_probability'] - 0.5
).abs()

display(cross_domain_errors['error_type'].value_counts().rename('errors').to_frame())
cross_domain_errors.sort_values('confidence', ascending=False).loc[
    :, ['error_type', 'spam_probability', 'text']
].head(5)

## Final comparison

In the initial seed-42 benchmark, DistilBERT improves SMS -> Enron F1 from 0.512 for TF-IDF to 0.557, but remains slightly below TextCNN at 0.563. Its in-domain-to-transfer gap is 0.416, and SMS -> Enron ROC-AUC is 0.480. This run does not remove the observed transfer gap; the 0.006 difference from TextCNN is too small to interpret without repeated runs.

In [ ]:
def f1_for(frame, train_domain, test_domain):
    return frame.query(
        'train_domain == @train_domain and test_domain == @test_domain'
    )['f1'].iloc[0]

baseline_cross_f1 = f1_for(baseline_results, 'sms', 'enron')
textcnn_cross_f1 = f1_for(textcnn_reference, 'sms', 'enron')
distilbert_in_domain_f1 = f1_for(distilbert_results, 'sms', 'sms')
distilbert_cross_f1 = f1_for(distilbert_results, 'sms', 'enron')
distilbert_transfer_gap = distilbert_in_domain_f1 - distilbert_cross_f1

print(f'TF-IDF SMS -> Enron F1:       {baseline_cross_f1:.3f}')
print(f'TextCNN SMS -> Enron F1:       {textcnn_cross_f1:.3f}')
print(f'DistilBERT SMS -> SMS F1:      {distilbert_in_domain_f1:.3f}')
print(f'DistilBERT SMS -> Enron F1:    {distilbert_cross_f1:.3f}')
print(f'DistilBERT transfer gap:       {distilbert_transfer_gap:.3f}')
print(
    'Change vs ML cross-domain:      '
    f'{distilbert_cross_f1 - baseline_cross_f1:+.3f}'
)
print(
    'Change vs TextCNN cross-domain: '
    f'{distilbert_cross_f1 - textcnn_cross_f1:+.3f}'
)

cross_domain_scores = {
    'TF-IDF + Logistic Regression': baseline_cross_f1,
    'TextCNN': textcnn_cross_f1,
    'DistilBERT': distilbert_cross_f1,
}
best_model = max(cross_domain_scores, key=cross_domain_scores.get)
print(f'Best SMS-to-Enron model by F1: {best_model}.')

## Scope and limitations

- Inputs are limited to 256 tokens, which affects long emails more than SMS messages.
- The displayed benchmark results come from one fixed split, seed, threshold, and configuration; they are not yet multi-seed estimates.
- The 0.5 threshold is not recalibrated for the target domain, so cross-domain precision and recall reflect calibration shift as well as representation quality.
- SMS and Enron differ in length, class balance, collection source, and writing style; the observed gap cannot be attributed to message format alone.
- Test labels are excluded from training and checkpoint selection. The locked robustness protocol keeps them out of future hyperparameter and threshold decisions.
- The test results have already been inspected, so they are treated as fixed confirmation benchmarks rather than pristine unseen holdouts. External data would be needed for a deployment claim.